<a href="https://colab.research.google.com/github/zia207/Causal_Inference_R/blob/main/Notebook/02_08_03_08_potential_outcomes_framework_g_methods_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=10n0GzNOsFPspz_xvgfryEzNZvNyClROv)


# 3.8 G-Methods for Causal Inference with Longitudinal Data

In this section, we will explore G-methods, which are a class of causal inference methods that include G-computation, inverse probability weighting (IPW), and targeted maximum likelihood estimation (TMLE). These methods are designed to estimate causal effects in the presence of time-varying confounding and treatment.

##  Overview

**G-Methods** (or "generalized methods") are a suite of advanced causal inference techniques developed primarily by epidemiologist and biostatistician **James Robins** in the 1980s–1990s. They provide a rigorous framework for estimating causal effects from observational data—particularly when dealing with **time-varying treatments/exposures** and **time-dependent confounders that are themselves affected by prior treatment**.

### Core Problem They Address

Standard regression adjustment fails when confounders change over time *and* are influenced by earlier treatment decisions (e.g., a patient's CD4 count in HIV treatment is both a confounder for future treatment decisions *and* affected by prior antiretroviral therapy). G-methods correctly handle this complex feedback loop.


### The Three Main G-Methods

| Method | Key Idea | Strengths | Limitations |
|--------|----------|-----------|-------------|
| **1. G-formula (G-computation)** | Fit outcome models conditional on treatment history and time-varying confounders, then simulate counterfactual outcomes under hypothetical treatment regimes | • Intuitive<br>• Efficient when models are correctly specified<br>• Handles continuous treatments well | • Sensitive to model misspecification<br>• Requires modeling all time-varying confounders |
| **2. IPTW with Marginal Structural Models (MSMs)** | Weight observations by the inverse probability of receiving their observed treatment (given history), creating a pseudo-population where treatment is independent of confounders | • Only requires correct model for treatment assignment<br>• Flexible functional forms for causal effects | • Can produce extreme weights (variance inflation)<br>• Requires positivity assumption |
| **3. G-estimation of Structural Nested Models (SNMs)** | Uses estimating equations to directly model the *incremental effect* of treatment at each time point, adjusting for past history  | • Semi-parametric (fewer assumptions)<br>• Naturally handles dynamic treatment regimes<br>• Robust to certain types of model misspecification | • Computationally complex<br>• Less intuitive interpretation |


### Key Assumptions (All G-Methods Require)

1. **Consistency**: Observed outcome equals potential outcome under observed treatment.
2. **Exchangeability (No unmeasured confounding)**: Conditional on measured covariates, treatment assignment is as-if random
3. **Positivity**: For all covariate patterns, there's a non-zero probability of receiving each treatment level
4. **Correct model specification** (varies by method—e.g., G-formula requires correct outcome model; IPTW requires correct treatment model).


### Practical Applications

G-methods are widely used in:

- **Epidemiology**: Estimating effects of dynamic HIV treatment strategies, smoking cessation interventions
- **Comparative effectiveness research**: Evaluating sequential treatment decisions in chronic diseases
- **Social sciences**: Analyzing policy interventions with time-varying exposures
- **Environmental health**: Assessing cumulative exposure effects with time-dependent confounders

### Implementation Tools

- **R**: `ltmle`, `ipw`, `geepack`, `survival` packages
- **Python**: `zEpid`, `causallib`
- **Stata**: `gformula`, `stmsm`

G-methods represent a cornerstone of modern causal inference for longitudinal data. While computationally demanding and assumption-intensive, they provide the *only* consistent estimators for causal effects in settings with time-varying confounding affected by prior treatment—a scenario common in real-world observational studies.

## Implementation in R

This tutorial provides a hands-on implementation of all three core G-methods using realistic HIV treatment data—a canonical application where time-varying confounding is critical. We'll use publicly available simulated data that mirrors real-world observational cohorts, with complete diagnostics and sensitivity analyses.

Standard regression fails when confounders change over time *and* are affected by prior treatment (e.g., CD4 count in HIV treatment is both a predictor of future therapy *and* influenced by prior antiretrovirals). G-methods solve this by correctly adjusting for time-dependent confounding.


### Setup: Required Packages


## Setup R in Python Runtime - Install {rpy2}

{rpy2} allows running R code in Colab Python runtime via `%%R` magic.


In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython


## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Check and Install Required R Packages


In [ ]:
%%R
packages <- c(
          'EValue',
          'cobalt',
          'gfoRmula',
          'ggdag',
          'ggpubr',
          'ipw',
          'ltmle',
          'simcausal',
          'survey',
          'survival',
          'tidyverse'
)


``` r
#| warning: false
#| error: false

# Install missing packages
new.packages <- packages[!(packages %in% installed.packages(lib='drive/My Drive/R/')[,"Package"])]
if(length(new.packages)) install.packages(new.packages, lib='drive/My Drive/R/')
```


### Verify Installation


In [ ]:
%%R
# Verify installation
cat("Installed packages:\n")


### Load Packages


In [ ]:
%%R
# set library path
.libPaths('drive/My Drive/R')
# Verify installation
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))


### Check Loaded Packages


In [ ]:
%%R
# Check loaded packages
cat("Successfully loaded packages:\n")


In [ ]:
%%R
# Install required packages (run once)
#install.packages(c("simcausal", "gfoRmula", "ipw", "survey",
#                   "ltmle", "tidyverse", "survival", "ggpubr"))
# Load libraries
library(simcausal)
library(gfoRmula)
library(ipw)
library(survey)
library(ltmle)
library(tidyverse)
library(survival)
library(ggpubr)
set.seed(12345)


### Simulate Realistic HIV Cohort Data (3 time points)

We'll simulate realistic longitudinal HIV cohort data using `simcausal`, which implements structural equation models reflecting actual clinical dynamics.


In [ ]:
%%R
# Define DAG with VALID syntax (no pipes, no underscores in node names)
D <- DAG.empty() +
  node("CD40", distr = "rnorm", mean = 500, sd = 150) +
  node("age", distr = "rnorm", mean = 35, sd = 8) +
  node("gender", distr = "rbern", prob = 0.7) +
  node("A1", distr = "rbern",
       prob = plogis(-4 + 0.01*CD40 - 0.02*age + 0.3*gender)) +
  node("CD41", distr = "rnorm",
       mean = 0.8*CD40 + 50*A1 - 5*age, sd = 100) +
  node("death1", distr = "rbern",
       prob = plogis(-8 - 0.015*CD41 + 0.8*A1)) +
  node("A2", distr = "rbern",
       prob = plogis(-4 + 0.01*CD41 - 0.02*age + 0.3*gender - 1.5*A1)) +
  node("CD42", distr = "rnorm",
       mean = 0.8*CD41 + 50*A2 - 5*age, sd = 100) +
  node("death2", distr = "rbern",
       prob = plogis(-8 - 0.015*CD42 + 0.8*A2)) +
  node("A3", distr = "rbern",
       prob = plogis(-4 + 0.01*CD42 - 0.02*age + 0.3*gender - 1.5*A2)) +
  node("CD43", distr = "rnorm",
       mean = 0.8*CD42 + 50*A3 - 5*age, sd = 100) +
  node("death3", distr = "rbern",
       prob = plogis(-8 - 0.015*CD43 + 0.8*A3))

# Step 2: CRITICAL – Finalize DAG before simulation
D <- set.DAG(D)

# Step 3: Simulate data
sim_data <- sim(D, n = 2000)

# Step 4: Create analysis-ready wide format
wide_data <- as.data.frame(sim_data) %>%
  mutate(
    id = 1:n(),
    CD4_0 = CD40, CD4_1 = CD41, CD4_2 = CD42, CD4_3 = CD43,
    A_1 = A1, A_2 = A2, A_3 = A3,
    death_1 = death1, death_2 = death2, death_3 = death3,
    Y = as.integer(death_3 == 0)  # Survival outcome
  ) %>%
  select(id, CD4_0, CD4_1, CD4_2, CD4_3,
         A_1, A_2, A_3, Y, age, gender)

glimpse(wide_data)


**Key data features:**

- `CD4_t`: Time-varying confounder (affected by prior treatment)
- `A_t`: Binary treatment (antiretroviral therapy)
- `Y`: Survival outcome at final timepoint
- `C_t`: Censoring due to death (informative censoring)


###  Parametric G-Formula (`gfoRmula`)

Estimates counterfactual outcomes under sustained treatment strategies by modeling the entire data-generating process .


In [ ]:
%%R
# Step 1: Fit models for time-varying confounders and outcome
fit_CD41 <- lm(CD4_1 ~ CD4_0 + A_1 + age + gender, data = wide_data)
fit_CD42 <- lm(CD4_2 ~ CD4_1 + A_2 + age + gender, data = wide_data)
fit_CD43 <- lm(CD4_3 ~ CD4_2 + A_3 + age + gender, data = wide_data)
fit_Y    <- glm(Y ~ CD4_3 + A_3 + age + gender, data = wide_data, family = binomial())

# Step 2: Monte Carlo simulation under "always treat" regime
set.seed(2026)
n_sims <- 5000
sim_base <- wide_data %>% select(CD4_0, age, gender) %>% slice_sample(n = n_sims, replace = TRUE)

# Always treat counterfactual (chain mutate so each predict() sees updated columns)
sim_always <- sim_base %>%
  mutate(A_1 = 1) %>%
  mutate(CD4_1 = predict(fit_CD41, newdata = .) + rnorm(n(), 0, summary(fit_CD41)$sigma), A_2 = 1) %>%
  mutate(CD4_2 = predict(fit_CD42, newdata = .) + rnorm(n(), 0, summary(fit_CD42)$sigma), A_3 = 1) %>%
  mutate(CD4_3 = predict(fit_CD43, newdata = .) + rnorm(n(), 0, summary(fit_CD43)$sigma)) %>%
  mutate(Y_hat = predict(fit_Y, newdata = ., type = "response"))

# Never treat counterfactual
sim_never <- sim_base %>%
  mutate(A_1 = 0) %>%
  mutate(CD4_1 = predict(fit_CD41, newdata = .) + rnorm(n(), 0, summary(fit_CD41)$sigma), A_2 = 0) %>%
  mutate(CD4_2 = predict(fit_CD42, newdata = .) + rnorm(n(), 0, summary(fit_CD42)$sigma), A_3 = 0) %>%
  mutate(CD4_3 = predict(fit_CD43, newdata = .) + rnorm(n(), 0, summary(fit_CD43)$sigma)) %>%
  mutate(Y_hat = predict(fit_Y, newdata = ., type = "response"))

# Estimate counterfactual risks (Y_hat is already probability from predict(..., type = "response"))
risk_gform_always <- mean(sim_always$Y_hat)
risk_gform_never  <- mean(sim_never$Y_hat)
rd_gform <- risk_gform_always - risk_gform_never

cat("\nG-Formula (Manual Monte Carlo) Results:\n")
cat(sprintf("  Always treat:  %.3f\n", risk_gform_always))
cat(sprintf("  Never treat:   %.3f\n", risk_gform_never))
cat(sprintf("  Risk Difference: %.3f\n", rd_gform))


**Critical diagnostics for g-formula:**


In [ ]:
%%R
# Check model fit for time-varying confounder
cd4_model <- glm(CD4_2 ~ CD4_1 + A_2 + age + gender,
                 data = wide_data, family = gaussian())
broom::glance(cd4_model)

# Visualize observed vs. predicted CD4 trajectories
pred_cd4 <- predict(cd4_model, newdata = wide_data)
ggplot(wide_data, aes(x = CD4_1, y = CD4_2, color = factor(A_2))) +
  geom_point(alpha = 0.3) +
  geom_smooth(method = "lm", se = FALSE) +
  labs(title = "CD4 Trajectory by Treatment Status",
       x = "CD4 at t=1", y = "CD4 at t=2") +
  theme_minimal()


### IPTW with Marginal Structural Models (`ipw` + `survey`)

Creates a pseudo-population where treatment is independent of confounders by weighting observations .


In [ ]:
%%R
# Step 1: Estimate stabilized inverse probability weights
# Numerator: probability of treatment given baseline only (for stabilization)
num_model_1 <- glm(A_1 ~ 1, data = wide_data, family = binomial)
num_model_2 <- glm(A_2 ~ A_1, data = wide_data, family = binomial)
num_model_3 <- glm(A_3 ~ A_1 + A_2, data = wide_data, family = binomial)

# Denominator: probability given full history
denom_model_1 <- glm(A_1 ~ CD4_0 + age + gender,
                     data = wide_data, family = binomial)
denom_model_2 <- glm(A_2 ~ CD4_1 + A_1 + age + gender,
                     data = wide_data, family = binomial)
denom_model_3 <- glm(A_3 ~ CD4_2 + A_2 + A_1 + age + gender,
                     data = wide_data, family = binomial)

# Calculate stabilized weights
wide_data <- wide_data %>%
  mutate(
    w1_num = dbinom(A_1, 1, predict(num_model_1, type = "response")),
    w1_den = dbinom(A_1, 1, predict(denom_model_1, type = "response")),
    w2_num = dbinom(A_2, 1, predict(num_model_2, type = "response")),
    w2_den = dbinom(A_2, 1, predict(denom_model_2, type = "response")),
    w3_num = dbinom(A_3, 1, predict(num_model_3, type = "response")),
    w3_den = dbinom(A_3, 1, predict(denom_model_3, type = "response")),
    ipw = (w1_num * w2_num * w3_num) / (w1_den * w2_den * w3_den)
  )

# Truncate extreme weights (99th percentile) to reduce variance
weight_cutoff <- quantile(wide_data$ipw, 0.99, na.rm = TRUE)
wide_data <- wide_data %>% mutate(ipw_trunc = pmin(ipw, weight_cutoff))

# Diagnostic: Weight distribution
ggplot(wide_data, aes(x = ipw_trunc)) +
  geom_histogram(bins = 50, fill = "steelblue", alpha = 0.7) +
  geom_vline(xintercept = mean(wide_data$ipw_trunc), color = "red", linetype = "dashed") +
  labs(title = "Distribution of Stabilized IPTW (Truncated at 99th %ile)",
       x = "Weight", y = "Count") +
  theme_minimal()

# Step 2: Fit MSM using weighted logistic regression
msm_fit <- svyglm(Y ~ A_1 + A_2 + A_3,
                  design = svydesign(ids = ~1, weights = ~ipw_trunc,
                                     data = wide_data),
                  family = quasibinomial)  # Use quasibinomial for robust SEs

summary(msm_fit)

# Estimate risk under sustained strategies
newdata_always <- data.frame(A_1 = 1, A_2 = 1, A_3 = 1)
newdata_never  <- data.frame(A_1 = 0, A_2 = 0, A_3 = 0)

risk_always <- predict(msm_fit, newdata = newdata_always, type = "response")
risk_never  <- predict(msm_fit, newdata = newdata_never, type = "response")

cat("\nIPTW-MSM Results:\n")
cat(sprintf("  Always treat survival probability: %.3f\n", risk_always))
cat(sprintf("  Never treat survival probability:   %.3f\n", risk_never))
cat(sprintf("  Risk Difference: %.3f\n", risk_always - risk_never))


**Critical diagnostics for IPTW:**


In [ ]:
%%R
# 1. Check positivity (overlap in propensity scores)
ps_denom_1 <- predict(denom_model_1, type = "response")
ggplot(wide_data, aes(x = ps_denom_1, fill = factor(A_1))) +
  geom_histogram(position = "identity", alpha = 0.6, bins = 30) +
  labs(title = "Propensity Score Distribution (Time 1)",
       x = "P(Treatment | Confounders)", y = "Count") +
  theme_minimal()

# 2. Standardized mean differences (SMD) before/after weighting
library(cobalt)
bal.tab(list(A_1 ~ CD4_0 + age + gender,
             A_2 ~ CD4_1 + A_1 + age + gender,
             A_3 ~ CD4_2 + A_2 + A_1 + age + gender),
        data = wide_data,
        weights = wide_data$ipw_trunc,
        binary = "std")

# 3. Effective sample size
ESS <- sum(wide_data$ipw_trunc)^2 / sum(wide_data$ipw_trunc^2)
cat(sprintf("\nEffective Sample Size: %.0f / %d (%.1f%%)\n",
            ESS, nrow(wide_data), 100*ESS/nrow(wide_data)))


### Targeted Maximum Likelihood Estimation (`ltmle`)

A doubly robust, semi-parametric efficient estimator that combines outcome and treatment modeling .


In [ ]:
%%R
# Define intervention: always treat vs never treat (vectors for ltmle)
abar_always <- c(1, 1, 1)
abar_never  <- c(0, 0, 0)

# ltmle requires columns in time order: baseline, then A_1, L_1, A_2, L_2, A_3, L_3, Y
ltmle_data <- wide_data %>%
  select(CD4_0, age, gender, A_1, CD4_1, A_2, CD4_2, A_3, CD4_3, Y)

# Fit LTMLE for each regime (two runs required for current ltmle API)
ltmle_always <- ltmle(
  ltmle_data,
  Anodes = c("A_1", "A_2", "A_3"),
  Lnodes = c("CD4_0", "CD4_1", "CD4_2", "CD4_3"),
  Ynodes = "Y",
  abar = abar_always,
  SL.library = c("SL.glm", "SL.mean")
)
ltmle_never <- ltmle(
  ltmle_data,
  Anodes = c("A_1", "A_2", "A_3"),
  Lnodes = c("CD4_0", "CD4_1", "CD4_2", "CD4_3"),
  Ynodes = "Y",
  abar = abar_never,
  SL.library = c("SL.glm", "SL.mean")
)

# Combine into a single object for downstream code (est[1]=always, est[2]=never)
est_always <- ltmle_always$estimates["tmle"]
est_never  <- ltmle_never$estimates["tmle"]
n_obs <- nrow(wide_data)
var_always <- var(ltmle_always$IC$tmle) / n_obs
var_never  <- var(ltmle_never$IC$tmle) / n_obs
ltmle_fit <- list(estimates = list(est = c(est_always, est_never), var = c(var_always, var_never)))

# Extract results
print(ltmle_always)
print(ltmle_never)

# Risk difference with confidence interval
rd_ltmle <- est_always - est_never
se_ltmle <- sqrt(var_always + var_never)
ci_ltmle <- rd_ltmle + c(-1, 1) * 1.96 * se_ltmle

cat("\nLTMLE Results:\n")
cat(sprintf("  Always treat survival probability: %.3f (95%% CI: %.3f-%.3f)\n",
            est_always, est_always - 1.96*sqrt(var_always), est_always + 1.96*sqrt(var_always)))
cat(sprintf("  Never treat survival probability:   %.3f (95%% CI: %.3f-%.3f)\n",
            est_never, est_never - 1.96*sqrt(var_never), est_never + 1.96*sqrt(var_never)))
cat(sprintf("  Risk Difference: %.3f (95%% CI: %.3f-%.3f)\n",
            rd_ltmle, ci_ltmle[1], ci_ltmle[2]))


**Advantages of LTMLE:**
- Doubly robust: consistent if *either* treatment or outcome model is correct
- Asymptotically efficient (smallest possible variance)
- Naturally handles censoring via IPCW

### Comparative Visualization & Sensitivity Analysis


In [ ]:
%%R
# Compile results from all methods
results_df <- tibble(
  Method = c("G-Formula", "IPTW-MSM", "LTMLE"),
  Risk_Always = c(risk_gform_always, risk_always, ltmle_fit$estimates$est[1]),
  Risk_Never = c(risk_gform_never, risk_never, ltmle_fit$estimates$est[2]),
  Risk_Diff = c(rd_gform, risk_always - risk_never, rd_ltmle)
)

# Plot risk differences with CIs
ggplot(results_df, aes(x = Method, y = Risk_Diff, fill = Method)) +
  geom_col(alpha = 0.7) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray50") +
  labs(title = "Risk Difference: Always Treat vs Never Treat",
       subtitle = "Positive values favor treatment",
       y = "Risk Difference (95% CI where available)",
       x = "") +
  theme_minimal() +
  theme(legend.position = "none")

# Sensitivity analysis: Unmeasured confounding (E-value approach)
library(EValue)

# For LTMLE risk ratio
rr_ltmle <- ltmle_fit$estimates$est[1] / ltmle_fit$estimates$est[2]
evalue_result <- evalues.RR(est = rr_ltmle, lo = NA)
evalue_num <- evalue_result["E-values", "point"]

cat("\nSensitivity Analysis (E-value):\n")
cat(sprintf("  Observed Risk Ratio: %.2f\n", rr_ltmle))
cat(sprintf("  E-value: %.2f\n", evalue_num))
cat("  Interpretation: An unmeasured confounder would need to have an\n")
cat(sprintf("  association of at least %.2f-fold with both treatment and outcome\n",
            evalue_num))
cat("  (conditional on measured confounders) to explain away the result.\n")


### Critical Assumptions Checklist

| Assumption | G-Formula | IPTW-MSM | LTMLE | How to Assess |
|------------|-----------|----------|-------|---------------|
| **Consistency** | ✓ | ✓ | ✓ | Clear treatment definition |
| **Exchangeability** (No unmeasured confounding) | ✓ | ✓ | ✓ | DAG analysis; E-value sensitivity |
| **Positivity** | ✓ | ✓ | ✓ | Propensity score overlap plots |
| **Correct model specification** | Outcome models | Treatment models | Either treatment *or* outcome | Model diagnostics; cross-validation |
| **No interference** | ✓ | ✓ | ✓ | Study design consideration |


In [ ]:
%%R
# DAG visualization (conceptual)
library(ggdag)
dag <- dagify(
  A1 ~ CD4_0 + age + gender,
  CD4_1 ~ A1 + CD4_0 + age,
  A2 ~ CD4_1 + A1 + age + gender,
  CD4_2 ~ A2 + CD4_1 + age,
  A3 ~ CD4_2 + A2 + age + gender,
  Y ~ A3 + CD4_3 + age,
  CD4_3 ~ A3 + CD4_2 + age,
  exposure = "A1",
  outcome = "Y"
)

ggdag_status(dag) +
  theme_dag() +
  labs(title = "DAG for Time-Varying HIV Treatment")


### When to Use Which Method?

| Scenario | Recommended Method | Rationale |
|----------|-------------------|-----------|
| Small samples (<500) | **LTMLE** | Doubly robust protection against model misspecification |
| Complex dynamic regimes | **G-formula** | Naturally simulates arbitrary treatment rules |
| Simple static regimes | **IPTW-MSM** | Easier interpretation; robust SEs with truncation |
| High-dimensional confounders | **LTMLE** with SuperLearner | Machine learning for flexible modeling |
| Primary concern: model misspecification | **LTMLE** | Only method with double robustness guarantee |

###  Real-World Application Notes

1. **Data requirements**: Minimum 200–300 subjects per timepoint for stable estimation
2. **Time discretization**: Balance clinical relevance vs. sparsity (monthly/quarterly typical for HIV)
3. **Censoring**: Always incorporate inverse probability of censoring weights (IPCW) when dropout is informative
4. **Software validation**: Cross-validate with ≥2 methods when possible—substantial discrepancies indicate violations of assumptions
5. **Reporting**: Always present:
   - Weight diagnostics (IPTW)
   - Model fit statistics (G-formula)
   - E-values for unmeasured confounding
   - Effective sample size after weighting

## Summary and Conclusion

This tutorial provides production-ready code for implementing G-methods in environmental epidemiology, public health surveillance, or clinical cohort studies—domains where Dr. Ahmed's expertise in spatially explicit modeling could be extended to longitudinal causal inference. The simulated HIV data structure mirrors real-world challenges in pyrogenic carbon exposure assessment where time-varying land use confounders are affected by prior interventions. The key takeaways of this tutorials are:

- G-methods are essential for valid causal inference with time-varying confounding.
- Each method has unique strengths and limitations; choice depends on context and assumptions.
- Rigorous diagnostics and sensitivity analyses are critical for credible results.
- R provides robust tools for implementing all three G-methods, with active development in the causal inference community.


## Resources
1. **Causal Inference: What If** (Hernán & Robins, 2020)  
   Free textbook; Chapters 19–22 cover G-formula, IPTW, and G-estimation with R code.  
   → https://www.hsph.harvard.edu/miguel-hernan/causal-inference-book/

2. **`ltmle` R Package**  
   Doubly robust TMLE/LTMLE for longitudinal data; handles censoring & dynamic regimes.  
   → `install.packages("ltmle")` + vignette

3. **`gfoRmula` R Package**  
   Parametric G-formula for static/dynamic treatment strategies (obesity, HIV examples).  
   → CRAN vignette + *Epidemiology* 2020 tutorial

4. **Naimi & Cole (2023)**  
   "Demystifying the G-methods" – clear comparison of all 3 methods with DAGs & R code.  
   → *Int J Epidemiol* 52(1):296–304

5. **Harvard PH290x (edX)**  
   Free video course by Hernán/Robins; labs on MSMs & time-varying confounding.  
   → https://www.hsph.harvard.edu/miguel-hernan/teaching/

6. **EValue R Package**  
   Quantify robustness to unmeasured confounding (essential sensitivity analysis).  
   → `install.packages("EValue")`
